Импорты


In [ ]:
import os
import ast
import logging
import requests
import psycopg2
from datetime import datetime, timedelta
from psycopg2.extras import execute_values

import gspread
from google.oauth2.service_account import Credentials
from collections import Counter
from dotenv import load_dotenv

import smtplib
import ssl
from email.message import EmailMessage

Конфигурация

In [ ]:
load_dotenv()  # читает .env из текущей папки

# ================= КОНФИГУРАЦИЯ =================
API_URL = "https://api.itresume.ru/statistics/"
API_PARAMS = {
    "client": os.getenv("API_CLIENT"),
    "client_key": os.getenv("API_CLIENT_KEY"),
    "start": "2023-04-01 00:00:00.000000",
    "end": "2023-04-30 23:59:59.999999"
}

DB_CONFIG = {
    "host": os.getenv("DB_HOST"),
    "port": os.getenv("DB_PORT"),
    "database": os.getenv("DB_NAME"),
    "user": os.getenv("DB_USER"),
    "password": os.getenv("DB_PASSWORD"),
}


LOG_DIR = os.getenv("LOG_DIR")

GOOGLE_CREDENTIALS_FILE = os.getenv("GOOGLE_CREDENTIALS_FILE")
SPREADSHEET_NAME = os.getenv("SPREADSHEET_NAME")

SMTP_SERVER = os.getenv("SMTP_SERVER")
SMTP_PORT = int(os.getenv("SMTP_PORT", 465))
SENDER_EMAIL = os.getenv("SENDER_EMAIL")
SENDER_PASSWORD = os.getenv("SENDER_PASSWORD")
RECIPIENT_EMAIL = os.getenv("RECIPIENT_EMAIL")

Логирование и очистка логов

In [ ]:
# ================= НАСТРОЙКА ЛОГИРОВАНИЯ =================
LOG_DIR = "logs"
LOG_FILE_BASE = "project_log"

# Создаём папку для логов, если её нет
os.makedirs(LOG_DIR, exist_ok=True)

# Удаляем логи старше 3 дней
now = datetime.now()
cutoff = now - timedelta(days=3)
for filename in os.listdir(LOG_DIR):
    filepath = os.path.join(LOG_DIR, filename)
    if not os.path.isfile(filepath):
        continue
    try:
        date_part = filename.replace(f"{LOG_FILE_BASE}_", "").replace(".log", "")
        file_date = datetime.strptime(date_part, "%Y-%m-%d")
        if file_date < cutoff:
            os.remove(filepath)
    except (ValueError, IndexError):
        continue

date_str = now.strftime("%Y-%m-%d")
log_filename = os.path.join(LOG_DIR, f"{LOG_FILE_BASE}_{date_str}.log")

logger = logging.getLogger("GraderProject")
logger.setLevel(logging.INFO)

if not logger.handlers:
    formatter = logging.Formatter(
        "%(asctime)s - %(levelname)s - %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S"
    )
    file_handler = logging.FileHandler(log_filename, encoding="utf-8")
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)

    console_handler = logging.StreamHandler()
    console_handler.setFormatter(formatter)
    logger.addHandler(console_handler)

logger.info("Скрипт работает")

Запрос к API

In [ ]:
logger.info(f"Начало скачивания данных из API. Параметры: {API_PARAMS}")

# Увеличиваем таймаут до 120 секунд, как в вашем рабочем скрипте
try:
    response = requests.get(API_URL, params=API_PARAMS, timeout=120)
    
    if response.status_code != 200:
        logger.error(f"Ошибка API. Status code: {response.status_code}. Ответ: {response.text[:500]}")
        raise SystemExit(1)
        
    raw_data = response.json()
    
    # ОЧЕНЬ ВАЖНАЯ ПРОВЕРКА: Убеждаемся, что пришел именно список
    if not isinstance(raw_data, list):
        logger.error(f"Неожиданный формат ответа API: ожидался list, получен {type(raw_data).__name__}. Ответ: {raw_data}")
        raise SystemExit(1)
        
    logger.info(f"Скачивание завершено. Получено записей: {len(raw_data)}")

except requests.exceptions.RequestException as e:
    logger.error(f"Ошибка подключения к API: {e}")
    raise SystemExit(1)
except ValueError as e:
    logger.error(f"Ошибка парсинга JSON от API: {e}")
    raise SystemExit(1)

Парсинг

In [ ]:
# def parse_passback_params(raw_str):
#     if not raw_str:
#         return None
#     try:
#         data = ast.literal_eval(raw_str)
#         if isinstance(data, dict):
#             return data
#     except (ValueError, SyntaxError):
#         pass
#     return None

Валидация и трансформация

In [ ]:
import ast
import logging
from datetime import datetime
from urllib.parse import urlparse

VALID_ATTEMPT_TYPES = {"run", "submit"}

def _parse_passback_params(raw) -> dict:
    """Безопасно преобразует строку вида "{'a': 1, 'b': 'x'}" в dict."""
    if not raw:
        return {}
    if isinstance(raw, dict):
        return raw
    try:
        parsed = ast.literal_eval(raw)
    except (ValueError, SyntaxError) as exc:
        logger.warning("Не удалось распарсить passback_params: %s", exc)
        return {}
    return parsed if isinstance(parsed, dict) else {}

def _is_valid_url(url) -> bool:
    if not isinstance(url, str) or not url:
        return False
    try:
        parsed = urlparse(url)
    except Exception:
        return False
    return parsed.scheme in ("http", "https") and bool(parsed.netloc)

def parse_record(record: dict) -> dict | None:
    """Валидирует и нормализует одну запись. Возвращает dict или None."""
    try:
        user_id = record.get("lti_user_id")
        if not isinstance(user_id, str) or not user_id.strip():
            logger.warning("Пропуск записи: некорректный lti_user_id=%r", user_id)
            return None

        passback = _parse_passback_params(record.get("passback_params"))

        oauth_consumer_key = passback.get("oauth_consumer_key", "")
        if not isinstance(oauth_consumer_key, str):
            logger.warning("Пропуск записи user_id=%s: некорректный oauth_consumer_key", user_id)
            return None

        lis_result_sourcedid = passback.get("lis_result_sourcedid")
        if not isinstance(lis_result_sourcedid, str) or not lis_result_sourcedid:
            logger.warning("Пропуск записи user_id=%s: отсутствует lis_result_sourcedid", user_id)
            return None

        lis_outcome_service_url = passback.get("lis_outcome_service_url")
        if not _is_valid_url(lis_outcome_service_url):
            logger.warning("Пропуск записи user_id=%s: некорректный lis_outcome_service_url", user_id)
            return None

        is_correct = record.get("is_correct")
        if is_correct is None or isinstance(is_correct, bool):
            pass
        elif isinstance(is_correct, int) and is_correct in (0, 1):
            is_correct = bool(is_correct)
        else:
            logger.warning("Пропуск записи user_id=%s: некорректный is_correct=%r", user_id, is_correct)
            return None

        attempt_type = record.get("attempt_type")
        if attempt_type not in VALID_ATTEMPT_TYPES:
            logger.warning("Пропуск записи user_id=%s: некорректный attempt_type=%r", user_id, attempt_type)
            return None

        created_at_raw = record.get("created_at")
        try:
            # Пытаемся с миллисекундами
            created_at = datetime.strptime(created_at_raw, "%Y-%m-%d %H:%M:%S.%f")
        except (TypeError, ValueError):
            try:
                # Фолбэк: пытаемся без миллисекунд (как было в вашем старом коде)
                created_at = datetime.strptime(created_at_raw, "%Y-%m-%d %H:%M:%S")
            except (TypeError, ValueError):
                logger.warning("Пропуск записи user_id=%s: некорректный created_at=%r", user_id, created_at_raw)
                return None

        return {
            "user_id": user_id,
            "oauth_consumer_key": oauth_consumer_key,
            "lis_result_sourcedid": lis_result_sourcedid,
            "lis_outcome_service_url": lis_outcome_service_url,
            "is_correct": is_correct,
            "attempt_type": attempt_type,
            "created_at": created_at,
        }

    except Exception as exc:
        logger.exception("Неожиданная ошибка при разборе записи: %s", exc)
        return None

def parse_records(records: list[dict]) -> list[dict]:
    logger.info("Начало обработки данных. Всего записей: %d", len(records))
    processed, skipped = [], 0
    for rec in records:
        parsed = parse_record(rec)
        if parsed is None:
            skipped += 1
        else:
            processed.append(parsed)
    logger.info("Обработка завершена. Валидных: %d, пропущено: %d", len(processed), skipped)
    return processed
# def validate_and_transform(record, logger):
#     try:
#         user_id = record.get("lti_user_id")
#         if not user_id or not isinstance(user_id, str):
#             logger.warning(f"Пропуск записи: невалидный user_id в {record}")
#             return None

#         passback = parse_passback_params(record.get("passback_params"))
#         if passback is None:
#             logger.warning(f"Пропуск записи: не удалось распарсить passback_params у user_id={user_id}")
#             return None

#         oauth_key = passback.get("oauth_consumer_key", "")
#         lis_sourcedid = passback.get("lis_result_sourcedid", "")
#         lis_url = passback.get("lis_outcome_service_url", "")

#         is_correct = record.get("is_correct")
#         if is_correct is not None and not isinstance(is_correct, bool):
#             logger.warning(f"Пропуск записи: is_correct не bool и не None у user_id={user_id}")
#             return None

#         attempt_type = record.get("attempt_type")
#         if attempt_type not in ("run", "submit"):
#             logger.warning(f"Пропуск записи: неизвестный attempt_type='{attempt_type}' у user_id={user_id}")
#             return None

#         created_at_str = record.get("created_at")
#         if not created_at_str:
#             logger.warning(f"Пропуск записи: отсутствует created_at у user_id={user_id}")
#             return None

#         try:
#             created_at = datetime.strptime(created_at_str, "%Y-%m-%d %H:%M:%S.%f")
#         except ValueError:
#             try:
#                 created_at = datetime.strptime(created_at_str, "%Y-%m-%d %H:%M:%S")
#             except ValueError:
#                 logger.warning(f"Пропуск записи: неверный формат created_at='{created_at_str}' у user_id={user_id}")
#                 return None

#         return (user_id, oauth_key, lis_sourcedid, lis_url, is_correct, attempt_type, created_at)

#     except Exception as e:
#         logger.error(f"Неожиданная ошибка при валидации записи: {e}. Запись: {record}")
#         return None

Валидация по всем записям

In [7]:
logger.info("Начало обработки и валидации данных...")
valid_tuples = []
skipped_count = 0

for record in raw_data:
    transformed = validate_and_transform(record, logger)
    if transformed:
        valid_tuples.append(transformed)
    else:
        skipped_count += 1

logger.info(f"Обработка завершена. Валидных записей: {len(valid_tuples)}. Пропущено: {skipped_count}")

# Быстрая проверка, что получилось
if valid_tuples:
    print("Пример валидной записи:")
    print(valid_tuples[0])

Пример валидной записи:
('d0aa33caaa564d569b05e4a360ca8ec9', '', 'course-v1:SkillFactory+MFTIDSLIGHT+2022_DEC:lms.skillfactory.ru-ee883216aeec45eb960e31c6b83a0c36:d0aa33caaa564d569b05e4a360ca8ec9', '', None, 'run', datetime.datetime(2023, 4, 1, 0, 1, 6, 590200))


Создание таблицы

In [8]:
conn = psycopg2.connect(**DB_CONFIG)
cursor = conn.cursor()

create_table_query = """
CREATE TABLE IF NOT EXISTS grader_attempts (
    id SERIAL PRIMARY KEY,
    user_id VARCHAR(255) NOT NULL,
    oauth_consumer_key VARCHAR(255),
    lis_result_sourcedid TEXT,
    lis_outcome_service_url TEXT,
    is_correct BOOLEAN,
    attempt_type VARCHAR(50),
    created_at TIMESTAMP,
    loaded_at TIMESTAMP DEFAULT NOW()
);
"""

cursor.execute(create_table_query)
conn.commit()
cursor.close()
conn.close()

logger.info("Таблица grader_attempts проверена/создана")
print("Таблица grader_attempts готова")

2026-11-22 22:04:08 - INFO - Таблица grader_attempts проверена/создана


Таблица grader_attempts готова


Заполнение БД

In [9]:
if not valid_tuples:
    logger.warning("Нет валидных данных для загрузки.")
else:
    logger.info("Начало заполнения базы данных...")
    conn = None
    try:
        conn = psycopg2.connect(**DB_CONFIG)
        cursor = conn.cursor()

        insert_query = """
            INSERT INTO grader_attempts 
            (user_id, oauth_consumer_key, lis_result_sourcedid, lis_outcome_service_url, is_correct, attempt_type, created_at)
            VALUES %s
        """
        execute_values(cursor, insert_query, valid_tuples)
        conn.commit()
        logger.info(f"Загрузка в БД завершена успешно. Вставлено строк: {len(valid_tuples)}")

    except psycopg2.Error as e:
        logger.error(f"Ошибка базы данных: {e}")
        if conn:
            conn.rollback()
    except Exception as e:
        logger.error(f"Неожиданная ошибка при загрузке в БД: {e}")
    finally:
        if conn:
            cursor.close()
            conn.close()
            logger.info("Соединение с БД закрыто.")

logger.info("Скрипт завершил работу")

2026-11-22 22:04:08 - INFO - Начало заполнения базы данных...
2026-11-22 22:04:14 - INFO - Загрузка в БД завершена успешно. Вставлено строк: 166953
2026-11-22 22:04:14 - INFO - Соединение с БД закрыто.
2026-11-22 22:04:14 - INFO - Скрипт завершил работу


Проверка результата в бд

In [10]:
conn = psycopg2.connect(**DB_CONFIG)
cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) FROM grader_attempts;")
print("Всего строк в таблице:", cursor.fetchone()[0])

cursor.execute("SELECT * FROM grader_attempts ORDER BY id DESC LIMIT 5;")
for row in cursor.fetchall():
    print(row)

cursor.close()
conn.close()

Всего строк в таблице: 166953
(166953, 'de0af5c91415b2462df249932d442617', '', 'course-v1:SkillFactory+SDA-3.0+FEB2021:lms.skillfactory.ru-ba69a652ec204848be25e4f74138ae7e:de0af5c91415b2462df249932d442617', 'https://lms.skillfactory.ru/courses/course-v1:SkillFactory+SDA-3.0+FEB2021/xblock/block-v1:SkillFactory+SDA-3.0+FEB2021+type@lti+block@ba69a652ec204848be25e4f74138ae7e/handler_noauth/grade_handler', None, 'run', datetime.datetime(2023, 4, 30, 23, 59, 21, 891535), datetime.datetime(2026, 11, 22, 22, 4, 8, 931834))
(166952, 'de0af5c91415b2462df249932d442617', '', 'course-v1:SkillFactory+SDA-3.0+FEB2021:lms.skillfactory.ru-ba69a652ec204848be25e4f74138ae7e:de0af5c91415b2462df249932d442617', 'https://lms.skillfactory.ru/courses/course-v1:SkillFactory+SDA-3.0+FEB2021/xblock/block-v1:SkillFactory+SDA-3.0+FEB2021+type@lti+block@ba69a652ec204848be25e4f74138ae7e/handler_noauth/grade_handler', None, 'run', datetime.datetime(2023, 4, 30, 23, 59, 7, 440061), datetime.datetime(2026, 11, 22, 22, 

Задание со звездочкой 1

In [11]:

GOOGLE_CREDENTIALS_FILE = os.getenv("GOOGLE_CREDENTIALS_FILE")
GOOGLE_SHEET_ID = os.getenv("GOOGLE_SHEET_ID")
GOOGLE_SHEET_NAME = os.getenv("GOOGLE_SHEET_NAME")

print("Credentials:", GOOGLE_CREDENTIALS_FILE)
print("Spreadsheet ID:", GOOGLE_SHEET_ID)
print("Worksheet:", GOOGLE_SHEET_NAME)

total_attempts = len(valid_tuples)
successful = sum(1 for row in valid_tuples if row[4] is True)
unique_users = len(set(row[0] for row in valid_tuples))
attempt_types = Counter(row[5] for row in valid_tuples)
report_date = datetime.now().strftime("%Y-%m-%d")

summary_row = [
    report_date,
    total_attempts,
    successful,
    unique_users,
    attempt_types.get("run", 0),
    attempt_types.get("submit", 0),
]

print("Агрегат за день:", summary_row)

gc = gspread.service_account(filename=GOOGLE_CREDENTIALS_FILE)


sh = gc.open_by_key(GOOGLE_SHEET_ID)

try:
    ws = sh.worksheet(GOOGLE_SHEET_NAME)
except gspread.WorksheetNotFound:
    ws = sh.add_worksheet(title=GOOGLE_SHEET_NAME, rows=1000, cols=10)

if not ws.get_all_values():
    ws.append_row([
        "Дата", "Всего попыток", "Успешных", "Уникальных юзеров", "run", "submit"
    ])

ws.append_row(summary_row)

print(f"Агрегат записан в Google Sheets: ID={GOOGLE_SHEET_ID}, лист='{GOOGLE_SHEET_NAME}'")

Credentials: praxis-effort-515517-f6-aa60999d610f.json
Spreadsheet ID: 1OdjlK-ledcE-WmYE1pdFMXt7_2_0ySeMDvVFUSKaQEw
Worksheet: Лист1
Агрегат за день: ['2026-11-22', 166953, 0, 2304, 166953, 0]
Агрегат записан в Google Sheets: ID=1OdjlK-ledcE-WmYE1pdFMXt7_2_0ySeMDvVFUSKaQEw, лист='Лист1'


Задание со звездочкой 2

In [12]:
import certifi

SMTP_SERVER = os.getenv("SMTP_SERVER")
SMTP_PORT = int(os.getenv("SMTP_PORT", 465))
SENDER_EMAIL = os.getenv("SENDER_EMAIL")
SENDER_PASSWORD = os.getenv("SENDER_PASSWORD")
RECIPIENT_EMAIL = os.getenv("RECIPIENT_EMAIL")    

try:
    msg = EmailMessage()
    msg["Subject"] = f"Отчёт по грейдеру за {report_date}"
    msg["From"] = SENDER_EMAIL
    msg["To"] = RECIPIENT_EMAIL

    body = f"""
    Данные успешно загружены в БД и Google Sheets.

    Агрегация за {report_date}:
    - Всего попыток: {total_attempts}
    - Успешных: {successful}
    - Уникальных пользователей: {unique_users}
    - run: {attempt_types.get("run", 0)}
    - submit: {attempt_types.get("submit", 0)}
    """
    msg.set_content(body)

    context = ssl.create_default_context(cafile=certifi.where())
    with smtplib.SMTP_SSL(SMTP_SERVER, SMTP_PORT, context=context) as server:
        server.login(SENDER_EMAIL, SENDER_PASSWORD)
        server.send_message(msg)

    logger.info(f"Email-уведомление отправлено на {RECIPIENT_EMAIL}")

except Exception as e:
    logger.error(f"Ошибка при отправке email: {e}")

2026-11-22 22:04:19 - INFO - Email-уведомление отправлено на zuanamir0511@gmail.com
